# Managing License and Monitoring Organizations

<center><img src="./img/monitoring-lizard.jpeg" /></center>

In [ ]:
from arcgis.gis import GIS
gis = GIS(profile="your_online_profile")

## Managing Licenses and Entitlements

#### List All Avaible Licenses

In [ ]:
lm = gis.admin.license

In [ ]:
license_list = lm.all()
license_list

#### Assigning a License to a User

In [ ]:
lic = license_list[-1]
lic

In [ ]:
lic.report

In [ ]:
user = gis.users.search("*")[4]
lic.assign(user.username, ["cityEngine"])

In [ ]:
lic.report

#### Removing an Entitlement

In [ ]:
lic.revoke(user.username, entitlements="*")

In [ ]:
lic.report

#### Getting License by Name

In [ ]:
pro_license = gis.admin.license.get('ArcGIS Pro')
pro_license

In [ ]:
pro_license.report

In [ ]:
%matplotlib inline
pro_license.plot()

#### Examining Assigned Licenses

In [ ]:
user = gis.users.me
pro_license.user_entitlement(user)

In [ ]:
pro_license.assign(user, entitlements=[
      'smpEuropeN',
      'smpLAmericaN',
      'smpMidEAfricaN',
      'smpNAmericaN',
      'desktopAdvN',
      'smpAsiaPacificN',
      'businessStdN',
      'imageAnalystN'])

In [ ]:
pro_license.user_entitlement(user)

## Generating Organizational Reports

- Generate the reports of the overall usage of the organizations.  
- Reports define organization usage metrics in one place for the day, week, or month.  
- Administrators can monitor who is using which services, consuming how much credits and storage within certain time period.

In [ ]:
import datetime as _dt
import pandas as pd

In [ ]:
date_time_str = '01/01/23'
then = _dt.datetime.strptime(date_time_str, '%d/%m/%y')
val = int(then.timestamp() * 1000)

### Creating Content Reports

In [ ]:
sun_dec10 = _dt.datetime(2026, 2, 1)
final_item = gis.users.me.report(
            report_type='content', duration="monthly", start_time=sun_dec10
    
        )
final_item

In [ ]:
# Read the item data into a temporary CSV file
csv_file = final_item.get_data()

# Read the CSV as a DataFrame
df = pd.read_csv(csv_file)

In [ ]:
df.sort_values(by=['View Counts'], ascending=False).head()

#### Looking for the Largest Sized Items

In [ ]:
df.sort_values(by=['File Storage Size'], ascending=False).head()[['Title','Item Type', 'File Storage Size']]

#### Understand Storage by User

In [ ]:
gb = df.groupby("Owner")['File Storage Size'].sum()
gb.nlargest(4).plot(kind='barh')

In [ ]:
final_item.delete(permanent=True)

## Credit Reporting

<center><img src="./img/credit-monitoring.jpg"/></center>

In [ ]:
import datetime as _dt
date_time_str = '1/2/2026'
then = _dt.datetime.strptime(date_time_str, '%d/%m/%Y')

monthly_item = gis.users.me.report(
            report_type='credits', duration="weekly", start_time=then
        )

#### Examine the Data

In [ ]:
import pandas as pd
# Read the item data into a temporary CSV file
csv_credits_file = monthly_item.get_data()

# Read the CSV as a DataFrame
df = pd.read_csv(csv_credits_file, skiprows=3)

In [ ]:
df.head()

In [ ]:
q = df['File Storage'] > 0
df[q]

In [ ]:
q = df['Geocoding'] > 0
df[q]

### Managing Credit Usage

- enable credit allocation to all users
- Credit budgeting is not enabled by default

In [ ]:
credit_mgr = gis.admin.credits
credit_mgr

#### Enabling Credit Management

In [ ]:
if credit_mgr.is_enabled == False:
    credit_mgr.enable()
credit_mgr.is_enabled # 

#### Setting Default Credit Limits

In [ ]:
credit_mgr.default_limit

In [ ]:
credit_mgr.default_limit = 600

In [ ]:
credit_mgr.default_limit

#### Allocating Credits to a User

- Assignment of credits beyond the default is some necessary

In [ ]:
credit_mgr.allocate(user, 10000)  # sets to a specific amount

In [ ]:
credit_mgr.allocate(user, -1)  # sets to a unlimited credits

## Working with Logs

### Portal Logs


- A record of events that occurred
- Used for monitoring and troubleshooting portal

#### Examples of Logs Incidents:

+ Installation and upgrade events, such as authorizing the software and creating the portal website
+ Publishing of services and items, such as hosted services, web maps, and data items
+ Content management events, such as sharing items, changing item ownership, and adding, updating, moving, and deleting items
+ Security events, such as users logging in to the portal, creating, deleting, and disabling users, creating and changing user roles, updating HTTP and HTTPS settings, import and export of security certificates, and updating the portal's identity store
+ Organization management events, such as adding and configuring groups, adding or removing users from a group, configuration of the gallery, basemaps, utility services, and federated servers, and configuring log settings and deleting logs
+ General events, such as updating the portal's search index and restarting the portal

In [ ]:
gis = GIS(profile='your_enterprise_profile')
logs = gis.admin.logs
logs

#### Log Settings

- Modify, update basic storage and save setting

In [ ]:
logs.settings

#### Querying Logs

In [ ]:
import datetime
import pandas as pd

In [ ]:
results = logs.query(start_time=datetime.datetime.now() - datetime.timedelta(days=20))

In [ ]:
results['logMessages'][:2]

In [ ]:
%matplotlib inline
df = pd.DataFrame(results['logMessages'])
df.type.value_counts().plot.bar()

#### Monitoring Service Usage

ArcGIS Server records various service statistics, such as total requests, average response time and timeouts. Administrators and publishers can use this information to monitor service activity to better understand how clients are using services. For example, monitoring server statistics help you answer questions such as:

- What is the total number of requests that my ArcGIS Server site handled during the past week?
- How was the service request load distributed during the past month?
- How are my services performing on an hourly basis?
- What was the maximum number of service instances used at any given time for a particular service?

In [ ]:
gis = GIS(profile='your_enterprise_profile', verify_cert=False, trust_env=True)
admin = gis.admin
servers = admin.servers.get("HOSTING_SERVER")
server = servers[0]

In [ ]:
usage = server.usage
usage

##### Using built-in report

In [ ]:
reports = usage.list()
reports

In [ ]:
for r in reports:
    print(r.properties['reportname'])

##### Querying maximum response times for the last 7 days

In [ ]:
data = reports[2].query()


In [ ]:
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
%matplotlib inline

In [ ]:
#store reponse times in Y axis
data_y = data['report']['report-data'][0][0]['data']

#convert dates to readable dates and store in X axis
data_x = [pd.to_datetime(datetime.fromtimestamp(d//1000)) \
          for d in data['report']['time-slices']]

df = pd.DataFrame(list(zip(data_x, data_y)), columns=["date", "count"])
q = df['count'].isnull() # change NaN values to 0
df.loc[q, 'count'] = 0
df.index = df['date']
df['count'] = df['count'] 

ax = df['count'].plot(kind='bar', x=df['date'])
ticklabels = ['']*len(df.index)
ticklabels[::4] = [item.strftime('%b %d') for item in df.index[::4]]
ax.xaxis.set_major_formatter(ticker.FixedFormatter(ticklabels))
ax.set_title('Maximum reponse time in the last 7 days')
ax.set_ylabel('Time in seconds')
plt.gcf().autofmt_xdate()
#plt.show()

In [ ]:
data_x = [pd.to_datetime(datetime.fromtimestamp(d//1000)) \
          for d in data['report']['time-slices']]
data_x